In [ ]:
import geopandas as gpd
from shapely import geometry

from dep_tools.grids import PACIFIC_EPSG

In [ ]:
# Configure the resolution of the grid
RESOLUTION = 20
DEPTH_LIMIT = -100

# Load the regions file, and select our region
regions = gpd.read_file("postcards.geojson")

In [ ]:
for region in regions.itertuples():
    print(f"Working on {region.name}")
    bathy = f"/Users/alex/Data/bathymetry/{region.abbrev}.gpkg"

    # Pick the datafile that matches the region in the below step
    data = gpd.read_file(bathy, bbox=region.geometry.boundary)

    # If a value is positive, make it negative
    data['depth'] = data['depth'].apply(lambda x: x if x < 0 else -x)

    # Filter out data that is very deep
    data = data[data['depth'] > DEPTH_LIMIT]

    data = data.to_crs(PACIFIC_EPSG)

    # Get minX, minY, maxX, maxY
    minX, minY, maxX, maxY = data.total_bounds

    # Create a fishnet
    x, y = (minX, minY)
    geom_array = []

    # Polygon Size
    square_size = 20
    while y <= maxY:
        while x <= maxX:
            geom = geometry.Polygon([(x,y), (x, y+square_size), (x+square_size, y+square_size), (x+square_size, y), (x, y)])
            geom_array.append(geom)
            x += square_size
        x = minX
        y += square_size

    fishnet = gpd.GeoDataFrame(geom_array, columns=['geometry']).set_crs(PACIFIC_EPSG)

    print(f"Created fishnet with {len(fishnet)} polygons")

    # Combine the fishnet with the bathymetry and get the average depth
    joined = fishnet.sjoin(data, how="right", predicate="intersects")

    non_null = joined[joined['depth'].notna()]

    # Get the mean, median and stdev depth and the geometry
    grouped = non_null.groupby('index_left')['depth'].agg(['mean', 'median', 'std'])

    # Join index_left to the fishnet
    fishnet['index_left'] = fishnet.index
    fishnet = fishnet.set_index('index_left')
    joined_fishnet = fishnet.join(grouped, how='inner')

    joined_fishnet["geometry"] = joined_fishnet["geometry"].centroid

    # Rename mean to depth
    joined_fishnet = joined_fishnet.rename(columns={"mean": "depth"})

    # Write out file
    joined_fishnet.to_file(f"data/{region.name}_{RESOLUTION}.gpkg", overwrite=True)

    print(f"Wrote {len(joined_fishnet)} points\n-")

print("Done")